# 04 — Modeling and Business Results

Compare three models with cross-validation, tune the best, evaluate on a held-out set, and translate model performance into business impact.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay,
    confusion_matrix, ConfusionMatrixDisplay,
)

from health_insurance_cross_sell.config import load_config
from health_insurance_cross_sell.features import model_matrix, prepare_features
from health_insurance_cross_sell.models import cross_validate_models, tune_best_model

sns.set_theme(style="whitegrid", palette="muted")
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

config = load_config(PROJECT_ROOT / "configs" / "project.toml")

## 1. Load and Prepare Data

In [ ]:
from health_insurance_cross_sell.data import load_training_frame

df = load_training_frame(config)
X, y, _ = model_matrix(df, config, training=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Positive rate train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

## 2. Cross-Validation Comparison (5-fold Stratified)

In [ ]:
print("Running 5-fold cross-validation on all candidate models...")
print("This may take a few minutes.")

cv_results = cross_validate_models(config)
cv_results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(cv_results))
width = 0.35
ax.bar(x - width/2, cv_results["roc_auc_mean"], width, label="ROC AUC",
       color="#5b8db8", yerr=cv_results["roc_auc_std"], capsize=4)
ax.bar(x + width/2, cv_results["avg_precision_mean"], width, label="Avg Precision",
       color="#e07b39", yerr=cv_results["avg_precision_std"], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(cv_results["model"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Cross-Validation Results — 5-Fold Stratified", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_cv_comparison.png", dpi=150)
plt.show()

In [ ]:
best_model_name = cv_results.iloc[0]["model"]
print(f"Best model by ROC AUC: {best_model_name}")

## 3. Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
print(f"Tuning {best_model_name} with RandomizedSearchCV (20 iterations, 5-fold CV)...")
search = tune_best_model(config, best_model_name)

print(f"\nBest CV ROC AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

In [ ]:
model = search.best_estimator_

## 4. Final Evaluation on Held-Out Test Set

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = model.predict(X_test)

roc_auc = roc_auc_score(y_test, y_proba)
avg_prec = average_precision_score(y_test, y_proba)

K = 20_000
order = np.argsort(y_proba)[::-1][:K]
y_test_reset = y_test.reset_index(drop=True)
precision_at_k = float(y_test_reset.iloc[order].mean())
baseline = float(y_test.mean())
lift_at_k = precision_at_k / baseline

positives_captured = float(y_test_reset.iloc[order].sum() / y_test.sum())

print(f"ROC AUC          : {roc_auc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Precision@20000  : {precision_at_k:.4f}")
print(f"Lift@20000       : {lift_at_k:.2f}")
print(f"% positives captured in top 20k: {positives_captured*100:.1f}%")

### 4.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Not interested", "Interested"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "04_confusion_matrix.png", dpi=150)
plt.show()

### 4.2 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax, name=best_model_name)
ax.plot([0, 1], [0, 1], "--", color="grey", label="Random")
ax.set_title("ROC Curve", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_roc_curve.png", dpi=150)
plt.show()

### 4.3 Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=ax, name=best_model_name)
ax.axhline(baseline, color="grey", linestyle="--", label=f"Baseline ({baseline:.2f})")
ax.set_title("Precision-Recall Curve", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_pr_curve.png", dpi=150)
plt.show()

### 4.4 Cumulative Gain Curve

In [ ]:
sorted_idx = np.argsort(y_proba)[::-1]
y_sorted = y_test_reset.iloc[sorted_idx].values

cumulative_positives = np.cumsum(y_sorted) / y_test.sum()
pct_customers = np.arange(1, len(y_sorted) + 1) / len(y_sorted)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pct_customers * 100, cumulative_positives * 100,
        color="#e07b39", linewidth=2, label=best_model_name)
ax.plot([0, 100], [0, 100], "--", color="grey", label="Random baseline")
ax.axvline(K / len(y_test) * 100, color="#5b8db8", linestyle=":",
           label=f"Top {K:,} ({K/len(y_test)*100:.0f}% of base)")
ax.set_xlabel("% Customers Contacted")
ax.set_ylabel("% Positives Captured")
ax.set_title("Cumulative Gain Curve", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_cumulative_gain.png", dpi=150)
plt.show()

print(f"Contacting top {K:,} = {K/len(y_test)*100:.0f}% of customers captures",
      f"{positives_captured*100:.1f}% of interested buyers")

### 4.5 Lift Curve

In [ ]:
window = max(1, len(y_sorted) // 200)
lift_values = []
pct_values = []
for i in range(window, len(y_sorted) + 1, window):
    prec = y_sorted[:i].mean()
    lift_values.append(prec / baseline)
    pct_values.append(i / len(y_sorted) * 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pct_values, lift_values, color="#e07b39", linewidth=2, label=best_model_name)
ax.axhline(1.0, color="grey", linestyle="--", label="Random (lift=1)")
ax.axvline(K / len(y_test) * 100, color="#5b8db8", linestyle=":",
           label=f"Top {K:,}")
ax.set_xlabel("% Customers Contacted")
ax.set_ylabel("Lift")
ax.set_title("Lift Curve", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "04_lift_curve.png", dpi=150)
plt.show()

### 4.6 Feature Importances

In [ ]:
final_estimator = model.named_steps["model"]
preprocessor    = model.named_steps["preprocess"]

if hasattr(final_estimator, "feature_importances_"):
    importances = final_estimator.feature_importances_
    try:
        feature_names = preprocessor.get_feature_names_out()
    except Exception:
        feature_names = [f"f{i}" for i in range(len(importances))]

    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.nlargest(15, "importance").sort_values("importance")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(imp_df["feature"], imp_df["importance"], color="#5b8db8")
    ax.set_title(f"Top 15 Feature Importances — {best_model_name}", fontsize=13)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(FIGURES / "04_feature_importances.png", dpi=150)
    plt.show()
elif hasattr(final_estimator, "coef_"):
    coefs = np.abs(final_estimator.coef_[0])
    try:
        feature_names = preprocessor.get_feature_names_out()
    except Exception:
        feature_names = [f"f{i}" for i in range(len(coefs))]

    imp_df = pd.DataFrame({"feature": feature_names, "importance": coefs})
    imp_df = imp_df.nlargest(15, "importance").sort_values("importance")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(imp_df["feature"], imp_df["importance"], color="#5b8db8")
    ax.set_title(f"Top 15 |Coefficients| — {best_model_name}", fontsize=13)
    ax.set_xlabel("|Coefficient|")
    plt.tight_layout()
    plt.savefig(FIGURES / "04_feature_importances.png", dpi=150)
    plt.show()
else:
    print(f"Model {best_model_name} does not expose feature importances or coefficients.")

## 5. Business Translation

In [ ]:
revenue_per_conversion = 1_500
random_conversions = int(baseline * K)
model_conversions  = int(precision_at_k * K)
extra_conversions  = model_conversions - random_conversions
extra_revenue      = extra_conversions * revenue_per_conversion

print("=" * 55)
print("  BUSINESS IMPACT — TOP 20,000 CONTACTS")
print("=" * 55)
print(f"  Random strategy   : {random_conversions:,} conversions")
print(f"  Model strategy    : {model_conversions:,} conversions")
print(f"  Additional gains  : {extra_conversions:,} conversions")
print(f"  Lift @ 20,000     : {lift_at_k:.2f}x")
print(f"  % positives captured: {positives_captured*100:.1f}%")
print(f"  Estimated extra revenue: USD {extra_revenue:,.0f}")
print("=" * 55)

## 6. Save Model Artefacts

In [ ]:
model_path   = PROJECT_ROOT / "models" / "model.joblib"
metrics_path = PROJECT_ROOT / "reports" / "metrics.json"
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_path)

metrics = {
    "model": best_model_name,
    "roc_auc": round(roc_auc, 4),
    "average_precision": round(avg_prec, 4),
    "precision_at_20000": round(precision_at_k, 4),
    "lift_at_20000": round(lift_at_k, 4),
    "pct_positives_captured_at_20000": round(positives_captured, 4),
}
metrics_path.write_text(json.dumps(metrics, indent=2))

print(f"Model saved to   : {model_path}")
print(f"Metrics saved to : {metrics_path}")
print(json.dumps(metrics, indent=2))